In [1]:
from tokenizers import ByteLevelBPETokenizer

tokenizer = ByteLevelBPETokenizer()

tokenizer.train(
    files=["/content/train_personality.csv"],
    vocab_size=30522,
    min_frequency=2,
    special_tokens=[
    "<s>",  #start of sentence
    "</s>", # end of sentence
    "<pad>", #pad
    "<unk>", #unkown
])

tokenizer.save_model("/content/sample_data")

['/content/sample_data/vocab.json', '/content/sample_data/merges.txt']

In [2]:
import  torch
import torch.nn as nn
import torch.nn.functional as F
from dataclasses import dataclass
from tokenizers import ByteLevelBPETokenizer
from torch.utils.data import DataLoader, Dataset
import pandas as pd
from tqdm import tqdm
device = "cuda" if torch.cuda.is_available() else "cpu"

from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence
from sklearn.model_selection import train_test_split
from torch.cuda.amp import GradScaler, autocast


class CuasalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.nemb % config.nhead == 0
        self.c_attn = nn.Linear(config.nemb, 3 * config.nemb)
        self.c_proj = nn.Linear(config.nemb, config.nemb)

        self.nhead = config.nhead
        self.nemb = config.nemb

    def forward(self, x):
        B, T, C = x.size()
        qkv = self.c_attn(x)
        q, k, v = qkv.split(self.nemb, dim=2)
        k = k.view(B, T, self.nhead, C // self.nhead).transpose(1, 2)
        q = q.view(B, T, self.nhead, C // self.nhead).transpose(1, 2)
        v = v.view(B, T, self.nhead, C // self.nhead).transpose(1, 2)

        y = F.scaled_dot_product_attention(q, k, v, is_causal=True)
        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.c_proj(y)
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.nemb, 4 * config.nemb)
        self.gelu = nn.GELU(approximate="tanh")
        self.c_proj = nn.Linear(4 * config.nemb, config.nemb)

    def forward(self, x):
        x = self.c_fc(x)
        x = self.gelu(x)
        x = self.c_proj(x)
        return x

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln_1 = nn.LayerNorm(config.nemb)
        self.attn = CuasalSelfAttention(config)
        self.ln_2 = nn.LayerNorm(config.nemb)
        self.mlp = MLP(config)

    def forward(self, x):
        x = x + self.attn(self.ln_1(x))
        x = x + self.mlp(self.ln_2(x))
        return x


@dataclass
class GPTConfig:
    nemb :int = 256
    nhead :int = 2
    nlayers :int = 2
    vocabsiz :int = 30522
    blocksiz :int = 128

# GPT model class
class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.tranformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocabsiz, config.nemb),
            wpe = nn.Embedding(config.blocksiz, config.nemb),
            h = nn.ModuleList([Block(config) for _ in range(config.nlayers)]),
            ln_f = nn.LayerNorm(config.nemb)
        ))

        self.lm_head = nn.Linear(config.nemb, config.vocabsiz, bias=False)

        self.tranformer.wte.weight = self.lm_head.weight

        # init params
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            std = 0.02
            if hasattr(module, 'NANOGPT_SCALE_INIT'):
                std *= (2 * self.config.n_layer) ** -0.5
            torch.nn.init.normal_(module.weight, mean=0.0, std=std)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, ix, target=None):
        B, T = ix.size()
        assert T <= self.config.blocksiz, f"Cannot forward sequence of length {T}, block size is only {self.config.blocksiz}"

        pos = torch.arange(0, T, dtype=torch.long, device=ix.device)
        posEmb = self.tranformer.wpe(pos)
        tokEmb = self.tranformer.wte(ix)
        x = posEmb + tokEmb

        for block in self.tranformer.h:
            x = block(x)

        x = self.tranformer.ln_f(x)
        logits = self.lm_head(x)
        loss = None
        if target is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), target.view(-1))

        return logits, loss

    def generate(self, ix, max_new_tokens):
        for _ in range(max_new_tokens):
            ix_cond = ix[:, -self.config.blocksiz:]  # Use only the last `blocksiz` tokens
            logits, _ = self(ix_cond)
            logits = logits[:, -1, :]  # Get logits of the last token
            probs = F.softmax(logits, dim=-1)  # Convert logits to probabilities
            ix_next = torch.multinomial(probs, num_samples=1)  # Sample next token
            ix = torch.cat((ix, ix_next), dim=-1)  # Append to sequence
        return ix

# TextDataset class
class TextDataset(Dataset):
    def __init__(self, csv_path, tokenizer, block_size):
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.data = self.process_csv(csv_path)

    def process_csv(self, csv_path):
        df = pd.read_csv(csv_path)
        contexts = df['Context'].tolist()
        responses = df['Response'].tolist()
        tokens = [
            self.tokenizer.encode(context + " " + response).ids
            for context, response in zip(contexts, responses)
        ]
        return tokens

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        tokens = self.data[idx]
        # Padding and truncation
        if len(tokens) > self.block_size:
            tokens = tokens[:self.block_size]
        else:
            tokens = tokens + [0] * (self.block_size - len(tokens))  # Pad with zeros
        inputs = torch.tensor(tokens[:-1], dtype=torch.long)
        targets = torch.tensor(tokens[1:], dtype=torch.long)
        return inputs, targets

# Initialize model, tokenizer, and datasets
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = GPTConfig()
model = GPT(config)
model = model.to(device)

# Initialize tokenizer
tokenizer = ByteLevelBPETokenizer("/content/sample_data/vocab.json", "/content/sample_data/merges.txt")

blocksiz = 128
train_dataset = TextDataset(csv_path="/content/train_personality.csv", tokenizer=tokenizer, block_size=blocksiz)
val_dataset = TextDataset(csv_path="/content/train_personality.csv", tokenizer=tokenizer, block_size=blocksiz)

def collate_fn(batch):
    inputs = [item[0] for item in batch]
    targets = [item[1] for item in batch]
    inputs = pad_sequence(inputs, batch_first=True, padding_value=0)
    targets = pad_sequence(targets, batch_first=True, padding_value=0)
    return inputs, targets

trainloader = DataLoader(train_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn)
valloader = DataLoader(val_dataset, batch_size=32, shuffle=False, collate_fn=collate_fn)

# Define checkpoint saving and loading
checkpoint_path = "train_model.pth"

# Train the model
optim = torch.optim.AdamW(model.parameters(), lr=3e-3)
scaler = GradScaler()
epochs = 30
start_epoch = 0

for epoch in range(start_epoch, epochs):
    model.train()
    train_loss = 0
    progress_bar = tqdm(trainloader, desc=f"Epoch {epoch + 1}/{epochs} (Train)")

    for input, target in progress_bar:
        input = input.to(device)
        target = target.to(device)

        with autocast():
            logits, loss = model(input, target)

        optim.zero_grad()
        scaler.scale(loss).backward()
        scaler.step(optim)
        scaler.update()

        train_loss += loss.item()
        progress_bar.set_postfix(loss=train_loss / (len(progress_bar) + 1))

    avg_train_loss = train_loss / len(trainloader)

    model.eval()
    val_loss = 0
    with torch.no_grad():
        val_bar = tqdm(valloader, desc=f"Epoch {epoch + 1}/{epochs} (Validation)")
        for val_input, val_target in val_bar:
            val_input = val_input.to(device)
            val_target = val_target.to(device)

            with autocast():
                val_logits, val_loss_batch = model(val_input, val_target)

            val_loss += val_loss_batch.item()
            val_bar.set_postfix(loss=val_loss / (len(val_bar) + 1))

    avg_val_loss = val_loss / len(valloader)
    print(f"Epoch {epoch + 1}/{epochs} | Validation Loss: {avg_val_loss:.4f} |  Training Loss: {avg_train_loss:.4f}")

    # Save checkpoint
    checkpoint = {
        "epoch": epoch + 1,
        "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optim.state_dict(),
        "scaler_state_dict": scaler.state_dict(),
        "train_loss": avg_train_loss,
        "val_loss": avg_val_loss,
    }
    torch.save(checkpoint, checkpoint_path)
    print(f"Checkpoint saved to {checkpoint_path}")

<ipython-input-2-40684e5e2fad>:194: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = GradScaler()
Epoch 1/30 (Train):   0%|          | 0/280 [00:00<?, ?it/s]<ipython-input-2-40684e5e2fad>:207: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1/30 (Validation):   0%|          | 0/280 [00:00<?, ?it/s]<ipython-input-2-40684e5e2fad>:228: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with autocast():
Epoch 1/30 (Validation): 100%|██████████| 280/280 [00:04<00:00, 61.95it/s, loss=4.34]


Epoch 1/30 | Validation Loss: 4.3555 |  Training Loss: 5.1240
Checkpoint saved to train_model.pth


Epoch 2/30 (Validation): 100%|██████████| 280/280 [00:04<00:00, 61.75it/s, loss=3.96]


Epoch 2/30 | Validation Loss: 3.9781 |  Training Loss: 4.1904
Checkpoint saved to train_model.pth


Epoch 3/30 (Validation): 100%|██████████| 280/280 [00:04<00:00, 58.30it/s, loss=3.68]


...

Epoch 28/30 | Validation Loss: 1.3901 |  Training Loss: 1.5412
Checkpoint saved to train_model.pth


Epoch 29/30 (Validation): 100%|██████████| 280/280 [00:04<00:00, 60.57it/s, loss=1.35]


Epoch 29/30 | Validation Loss: 1.3523 |  Training Loss: 1.5045
Checkpoint saved to train_model.pth


Epoch 30/30 (Validation): 100%|██████████| 280/280 [00:04<00:00, 59.87it/s, loss=1.32]


Epoch 30/30 | Validation Loss: 1.3198 |  Training Loss: 1.4679
Checkpoint saved to train_model.pth


In [3]:


def load_checkpoint(model, optimizer, checkpoint_path):
    checkpoint = torch.load(checkpoint_path)
    model.load_state_dict(checkpoint["model_state_dict"])
    optimizer.load_state_dict(checkpoint["optimizer_state_dict"])
    print(f"Checkpoint loaded from {checkpoint_path}, Epoch: {checkpoint['epoch']}")
    return checkpoint["epoch"]

class TextDataset1(Dataset):
    def __init__(self, csv_path, tokenizer, block_size):
        """
        Args:
        - csv_path: Path to the labeled dataset in CSV format.
        - tokenizer: The tokenizer object used for encoding text.
        - block_size: Maximum sequence length for context-response pairs.
        """
        self.tokenizer = tokenizer
        self.block_size = block_size
        self.data = self.process_csv(csv_path)

    def process_csv(self, csv_path):
        """
        Reads the CSV file and tokenizes context-response pairs.
        """
        df = pd.read_csv(csv_path)
        contexts = df['Context'].tolist()
        responses = df['Response'].tolist()
        tokens = [
            self.tokenizer.encode(f"<s> {context} </s> {response} </s>").ids
            for context, response in zip(contexts, responses)
        ]
        return tokens

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        """
        Prepares input and target sequences for training.
        """
        tokens = self.data[idx]
        # Truncate or pad the sequence to block_size
        if len(tokens) > self.block_size:
            tokens = tokens[:self.block_size]
        else:
            tokens = tokens + [0] * (self.block_size - len(tokens))  # Pad with zeros

        # Prepare input and target tensors
        inputs = torch.tensor(tokens[:-1], dtype=torch.long)
        targets = torch.tensor(tokens[1:], dtype=torch.long)
        return inputs, targets
# Fine-Tune Dataset Preparation
fine_tune_csv_path = "/content/finetune_personality.csv"  # Replace with your fine-tune dataset path
fine_tune_dataset = TextDataset1(csv_path=fine_tune_csv_path, tokenizer=tokenizer, block_size=config.blocksiz)

fine_tune_loader = DataLoader(
    fine_tune_dataset, batch_size=32, shuffle=True, collate_fn=collate_fn
)

# Load Pretrained Model and Optimizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
config = GPTConfig()
model = GPT(config).to(device)
optimizer = torch.optim.AdamW(model.parameters(), lr=1e-4)  # Adjust learning rate if needed

# Load Checkpoint
checkpoint_path = "/content/train_model.pth"  # Path to your pretrained model checkpoint
start_epoch = load_checkpoint(model, optimizer, checkpoint_path)

# Fine-Tuning Loop
epochs = 20  # Adjust based on your needs
for epoch in range(start_epoch, start_epoch + epochs):
    model.train()
    train_loss = 0
    progress_bar = tqdm(fine_tune_loader, desc=f"Fine-Tuning Epoch {epoch + 1}")

    for inputs, targets in progress_bar:
        inputs, targets = inputs.to(device), targets.to(device)

        optimizer.zero_grad()
        logits, loss = model(inputs, targets)
        loss.backward()
        optimizer.step()

        train_loss += loss.item()
        progress_bar.set_postfix(loss=train_loss / (len(progress_bar) + 1))

    avg_train_loss = train_loss / len(fine_tune_loader)
    print(f"Epoch {epoch + 1}, Fine-Tuning Loss: {avg_train_loss:.4f}")

# Save Fine-Tuned Model
fine_tuned_model_path = "fine_tuned_model.pth"
torch.save({
    "epoch": epoch + 1,
    "model_state_dict": model.state_dict(),
    "optimizer_state_dict": optimizer.state_dict(),
}, fine_tuned_model_path)
print(f"Fine-tuned model saved to {fine_tuned_model_path}")


<ipython-input-3-e584ba421231>:2: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Checkpoint loaded from /content/train_model.pth, Epoch: 30


Fine-Tuning Epoch 31: 100%|██████████| 280/280 [00:22<00:00, 12.18it/s, loss=1.59]


Epoch 31, Fine-Tuning Loss: 1.5917


Fine-Tuning Epoch 32: 100%|██████████| 280/280 [00:23<00:00, 11.79it/s, loss=1.24]


...

Epoch 48, Fine-Tuning Loss: 0.7981


Fine-Tuning Epoch 49: 100%|██████████| 280/280 [00:24<00:00, 11.26it/s, loss=0.779]


Epoch 49, Fine-Tuning Loss: 0.7820


Fine-Tuning Epoch 50: 100%|██████████| 280/280 [00:24<00:00, 11.26it/s, loss=0.762]


Epoch 50, Fine-Tuning Loss: 0.7651
Fine-tuned model saved to fine_tuned_model.pth


In [6]:
import torch
import torch.nn.functional as F
from tokenizers import ByteLevelBPETokenizer

# Load the fine-tuned model and tokenizer
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

# Load the tokenizer
tokenizer = ByteLevelBPETokenizer(
    "/content/sample_data/vocab.json",
    "/content/sample_data/merges.txt"
)

# Load the fine-tuned model
checkpoint_path = "/content/fine_tuned_model.pth"
config = GPTConfig()
model = GPT(config).to(device)
checkpoint = torch.load(checkpoint_path)
model.load_state_dict(checkpoint["model_state_dict"])
model.eval()

# Chatting Function
def chat_with_model(prompt, max_new_tokens=50):

    # Tokenize the input prompt
    input_ids = tokenizer.encode(f"<s> {prompt} </s>").ids
    input_tensor = torch.tensor([input_ids], dtype=torch.long).to(device)

    # Generate response
    with torch.no_grad():
        output_ids = model.generate(input_tensor, max_new_tokens=max_new_tokens)

    # Decode the generated tokens
    output_text = tokenizer.decode(output_ids[0].tolist(), skip_special_tokens=True)

    # Return only the response part, not including the prompt
    response = output_text[len(prompt):].strip()
    return response
import builtins
# Start a chat session
def start_chat():
    """
    Starts an interactive chat session with the model.
    """
    print("Chatbot: Hi! How can I help you today? (type 'exit' to quit)")
    while True:
        user_input = builtins.input("You: ")
        if user_input.lower() == "exit":
            print("Chatbot: Goodbye!")
            break
        response = chat_with_model(user_input)
        print(f"Chatbot: {response}")

# Start chatting
start_chat()


<ipython-input-6-e6252b85038e>:18: FutureWarning: You are using `torch.load` with `weights_only=False` (the current default value), which uses the default pickle module implicitly. It is possible to construct malicious pickle data which will execute arbitrary code during unpickling (See https://github.com/pytorch/pytorch/blob/main/SECURITY.md#untrusted-models for more details). In a future release, the default value for `weights_only` will be flipped to `True`. This limits the functions that could be executed during unpickling. Arbitrary objects will no longer be allowed to be loaded via this mode unless they are explicitly allowlisted by the user via `torch.serialization.add_safe_globals`. We recommend you start setting `weights_only=True` for any use case where you don't have full control of the loaded file. Please open an issue on GitHub for any issues related to this experimental feature.
  checkpoint = torch.load(checkpoint_path)


Chatbot: Hi! How can I help you today? (type 'exit' to quit)
You: hi, how are you.
Chatbot: you. </s> Cponse: i was born on christmas day. i get sleepy. i stayed back to walmart manager on a been training for a marathon. </s> Response: hi . how is ur weekend ? ?
You: so, what you've been up to.?
Chatbot: to.? </s>onse: not a fan of obsession with pens. i love to be travel. </s> Response: hello there how are you ?
hi me a little sad because i never have energy of my plane skin
haha
You: hi, how's your day going.?
Chatbot: ng.? </s> Conse: my hobbies are read. i am getting married next week. i do not think my girlfriend understands. listen to linking park , i am excellent at math. </s> Response: hello anyway how is
You: so, what are you planing to do today.?
Chatbot: ay.? </s>onseonse: i am never still sure where its. i like to travel. i love the world taking night after work after work i am looking forward to retiring. </s> Response: thinking this new career cops .
You: hi, how are you 

# So seems like the fine tune quit work,
  - But i dont quite understand why is this Additonal like tokens are being generated.
    + EX:
       -  You: hi, how are you.
            
       - Chatbot: you. </s> **Cponse: i was born on christmas day. i get sleepy. i stayed back to walmart manager on a been training for a marathon.** </s> Response: hi . how is ur weekend ? ?
    + you can see these tokenes between </s> tokens..

# Definitly Need to Train More..
# But im happy..